In [1]:
is_interactive = True

In [2]:
import os
import ipynbname
from pathlib import Path

nb_path = ipynbname.path()
print(nb_path)
#os.chdir(os.path.dirname(nb_path))
os.chdir('/home/anandas/face_recognition')
print(os.getcwd())


/home/anandas/demo-jvsc-e1f4ef89-6d1f-440a-95d7-47ecdf2772284941214a-d076-4fdd-882d-4b901c5baeae.ipynb
/home/anandas/face_recognition


In [3]:
import os
from pathlib import Path
from sqlalchemy.orm import declarative_base
from face_recognition.face_recogniser import FaceRecognizer


In [4]:
import sqlalchemy as sa
from sqlalchemy import create_engine
from sqlalchemy.orm import declarative_base, sessionmaker
from types import SimpleNamespace
def create_db(path,preserve_past:bool = True ):
    engine = create_engine(f"sqlite:///{path}", echo=True)
    SessionLocal = sessionmaker(bind=engine)
    session = SessionLocal()
    db = SimpleNamespace(
        session=session, 
        engine=engine,
        Column=sa.Column, 
        Integer=sa.Integer, 
        String=sa.String, 
        Boolean=sa.Boolean, 
        ForeignKey=sa.ForeignKey
        )  
    if not preserve_past:
        metadata = sa.MetaData() 
        inspector = sa.inspect(engine) 
        for table_name in FaceRecognizer.tables(): 
            if table_name in inspector.get_table_names(): 
                table = sa.Table(table_name, metadata, autoload_with=engine) 
                table.drop(engine)
    return db
    

In [5]:
import lancedb
def create_vector_db(path, preserve_past:bool = True ):
    db = lancedb.connect(path)
    if not preserve_past:
        for table_name in FaceRecognizer.vector_tables(): 
            if table_name in db.table_names(): 
                tbl = db.open_table(table_name)
                tbl.delete("true")
    return db

In [6]:
def setup_face_dir(face_dir: str, preserve_past: bool = True):
    os.makedirs(face_dir, exist_ok=True)
    if not preserve_past:
        for filename in os.listdir(face_dir):
            file_path = os.path.join(face_dir, filename)
            if os.path.isfile(file_path) or os.path.islink(file_path):
                os.unlink(file_path)  # remove file or symlink
    return face_dir

In [7]:
def available_faces(path: str):
    return [
        (file.stem.split("_")[0], str(file))
        for file in Path(path).rglob("*")
        if file.suffix.lower() in (".png", ".jpg", ".jpeg")
    ]

### Main module


In [8]:
rebuild_store = True
preserve_past = not rebuild_store

In [9]:
db = create_db("face_store.db", preserve_past=preserve_past)
vectordb = create_vector_db("face_vector_store.db", preserve_past=preserve_past)
Base = declarative_base()
face_dir = setup_face_dir(face_dir="face_images", preserve_past=preserve_past)

recogniser = FaceRecognizer( db=db, dbModel=Base, vectordb=vectordb, face_dir=face_dir , is_interactive = is_interactive)
Base.metadata.create_all(db.engine)

2025-08-20 15:11:05,716 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-08-20 15:11:05,716 INFO sqlalchemy.engine.Engine SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite~_%' ESCAPE '~' ORDER BY name
2025-08-20 15:11:05,717 INFO sqlalchemy.engine.Engine [raw sql] ()
2025-08-20 15:11:05,718 INFO sqlalchemy.engine.Engine ROLLBACK
2025-08-20 15:11:05,719 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-08-20 15:11:05,720 INFO sqlalchemy.engine.Engine PRAGMA main.table_xinfo("person")
2025-08-20 15:11:05,720 INFO sqlalchemy.engine.Engine [raw sql] ()
2025-08-20 15:11:05,721 INFO sqlalchemy.engine.Engine SELECT sql FROM  (SELECT * FROM sqlite_master UNION ALL   SELECT * FROM sqlite_temp_master) WHERE name = ? AND type in ('table', 'view')
2025-08-20 15:11:05,722 INFO sqlalchemy.engine.Engine [raw sql] ('person',)
2025-08-20 15:11:05,723 INFO sqlalchemy.engine.Engine PRAGMA main.foreign_key_list("person")
2025-08-20 15:11:05,723 INFO sqlalchemy.engine.Engine 

In [10]:

if rebuild_store:
    faces = available_faces(
            "/home/anandas/demos/degirum_hailo_examples/assets/Friends_dataset"
        )
    print(faces)
    

[('Ross', '/home/anandas/demos/degirum_hailo_examples/assets/Friends_dataset/Ross_2.jpg'), ('Joey', '/home/anandas/demos/degirum_hailo_examples/assets/Friends_dataset/Joey_2.jpg'), ('Phoebe', '/home/anandas/demos/degirum_hailo_examples/assets/Friends_dataset/Phoebe_3.jpg'), ('Monica', '/home/anandas/demos/degirum_hailo_examples/assets/Friends_dataset/Monica_4.png'), ('Phoebe', '/home/anandas/demos/degirum_hailo_examples/assets/Friends_dataset/Phoebe_2.jpg'), ('Ross', '/home/anandas/demos/degirum_hailo_examples/assets/Friends_dataset/Ross_3.jpg'), ('Chandler', '/home/anandas/demos/degirum_hailo_examples/assets/Friends_dataset/Chandler_1.jpg'), ('Chandler', '/home/anandas/demos/degirum_hailo_examples/assets/Friends_dataset/Chandler_3.jpg'), ('Monica', '/home/anandas/demos/degirum_hailo_examples/assets/Friends_dataset/Monica_2.jpg'), ('Joey', '/home/anandas/demos/degirum_hailo_examples/assets/Friends_dataset/Joey_1.jpg'), ('Rachel', '/home/anandas/demos/degirum_hailo_examples/assets/Frien

In [18]:
if rebuild_store:
    recogniser.register_faces(faces)

ValueError: Number of rows must be a positive integer, not 0

<Figure size 1600x0 with 0 Axes>